## Creando una Vista hardcodead

Views - Vistas
- Basadas en funciones
- Basadas en clases

### Crear una aplicación en Django

python manage.py startapp <nombre_de_la_app>

python manage.py startapp ecommerce

- funciones en minúsculas
- clases comienzan con mayúsculas

### ecommerce/views.py

In [ ]:
from django.shortcuts import render
from django.http import HttpResponse

# Create your views here.
def home(request):
    return HttpResponse("<h1>Hola Mundo</h1>")

def home(request):
    html = """
    <!DOCTYPE html>
    <html>
        <head>
            <style>
                h1 {color: blue}
            </style>
        </head>

        <body>
            <h1>Hola Mundo</h1>
        </body>
    </html>
    """
    return HttpResponse(html)

### ecommerce/urls.py

In [ ]:
from django.urls import path

from ecommerce import views

urlpatterns = [
    path("", views.home, name="home"),
]

### config/urls.py

In [ ]:
from django.contrib import admin
from django.urls import include
from django.urls import path

urlpatterns = [
    path("up/", include("up.urls")),
    path("", include("pages.urls")),
    path("ecommerce/", include("ecommerce.urls")), #<---------

    path("admin/", admin.site.urls),
]

## Respuesta http y redireccionamiento

In [ ]:
middleware : procesos intermedios entre un request y un response

## CRUD y Vistas

Forma dinámica de usar las vistas

CRUD - Create, Retrieve, Update, Delete

- crear modelos
- Agregar la app a INSTALLED_APPS
- Crear y aplicar migraciones
- Agregar admin

### ecommerce/model.py

In [ ]:
from django.db import models

# Create your models here.

class ProductModel(model.Model):
    title = models.TextField()
    price = model.FloatField()

### config/settings.py

...

INSTALLED_APPS = [
    "pages.apps.PagesConfig",
    "ecommerce.apps.EcommerceConfig",
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",

........

## Correr migraciones

- python manage.py makemigrations
- python manage.py migrate

### Registrar modelos en admin.py

from django.contrib import admin

from .models import ProductModel

admin.site.register(ProductModel)

## Crear super user

python manage.py createsuperuser

admin1234
Test1234

### Tipos básicos de vistas

- List view : usuarios creados
- Create view : insertar datos para crear usuarios
- Retrieve and Update view : consultar y actualizar
- Delete view : eliminar usuario

## ecommerce/views.py

In [ ]:
from .models import ProductModel

def product_model_list_view(request):
    queryset = ProductModel.objects.all()
    print(queryset)
    return HttpResponse("ecommerce personalizado")

## Usando templates

### ecommerce/views.py

In [ ]:
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .models import ProductModel

def product_model_list_view(request):
    queryset = ProductModel.objects.all()
    print(queryset)
    template = "ecommerce/list-view.html"
    context = {}
    return render(request, template, context)

### templates/ecommerce/list-view.html

In [ ]:
<h1>
Vista de listado
</h1>

## Usando el contexto

### ecommerce/views.py

In [ ]:
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .models import ProductModel

def product_model_list_view(request):
    queryset = ProductModel.objects.all()
    print(queryset)
    template = "ecommerce/list-view.html"
    context = {
        "products": queryset #<--------
    }
    return render(request, template, context)

### templates/ecommerce/list-view.html

In [ ]:
<h1>
Vista de listado
</h1>

{% for product in products %}
    <li> 
        {{ product.title }} {{ products.price }}
    </li>
{% endfor %}

## Proteger endpoints

### ecommerce/list-view-public.html

In [ ]:
<h1>
    Un ecommerce genial, registrate
</h1>

## Vista de Detalle

### ecommerce/view.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.shortcuts import render, get_object_or_404 # <----------------
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .models import ProductModel

def product_model_detail_view(request, product_id):
    instance = get_object_or_404(ProductModel, id=product_id)
    context = {
        "product":instance
    }
    template = "ecommerce/detail-view.html"
    return render(request, template, context)

#..... otras templates......

### ecommerce/urls.py

In [ ]:
from django.urls import path

from ecommerce import views

urlpatterns = [
    path("", views.product_model_list_view, name="list"),
    path("<int:product_id>", views.product_model_detail_view, name="detail")
]

### ecommerce/templates/ecommerce/detail-view.html

In [ ]:
<h1>
    {{ product.title }}
    {{ product.price }}
</h1>

### templates/ecommerce/list-view.html

In [ ]:
<h1>
    Vista de listado
    </h1>
    
    {% for product in products %}
        <li> 
            <a href="/ecommerce/{{ product.id }}"> {{ product.title}}</a>
        </li>
    {% endfor %}

## Vista de Creación

### ecommerce/views.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.shortcuts import render, get_object_or_404
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .forms import ProductModelForm
from .models import ProductModel

def product_model_create_view(request):
    form = ProductModelForm(request.POST or None)
    if form.is_valid():
        instance = form.save(commit=False)
        instance.save()
        messages.success(request, "Producto creado con exito")
        return HttpResponseRedirect("/ecommerce/{product_id}".format(product_id=instance.id))
    context = {
        "form":form
    }
    template = "ecommerce/create-view.html"
    return render{request, template, context}


### ecommerce/forms.py

In [ ]:
from django import forms

from .models import ProductModel

class ProductModelForm(forms.ModelForm):
    class Meta:
        model = ProductModel
        fields = [
            "title",
            "price"
        ]

### templates/ecommerce/messagesd.html

In [ ]:
{% if messages %}
<ul class="messages">
    {% for message in messages %}
    <li{% if message.tags %} class="{{ message.tags }}"{% endif %}>{{ message }}</li>
    {% endfor %}
</ul>
{% endif %}

### templates/ecommerce/create-view.html

In [ ]:
{% include "ecommerce/messages.html" %}

<h1>
    Crear un nuevo producto
</h1>

<form method="POST" action="">
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Crear">
</form>

In [ ]:
NB : Agregar en todos los templates
    
{% include "ecommerce/messages.html" %}

### ecommerce/urls.py

In [ ]:
from django.urls import path

from ecommerce import views

urlpatterns = [
    path("", views.product_model_list_view, name="list"),
    path("<int:product_id>", views.product_model_detail_view, name="detail"),
    path("create", views.product_model_create_view, name="create"), #<----------------
]


## Vista de Actualización

### ecommerce/views.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.shortcuts import render, get_object_or_404
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .forms import ProductModelForm
from .models import ProductModel

def product_model_update_view(request, product_id=None):
    instance = get_object_or_404(ProductModel, id=product_id)
    form = ProductModelForm(request.POST or None, instance=instance)
    if form.is_valid():
        instance = form.save(commit=False)
        instance.save()
        messages.success(request, "Producto actualizado con exito")
        return HttpResponseRedirect("/ecommerce/{product_id}".format(product_id=instance.id))
    context = {
        "form":form
    }
    template = "ecommerce/update-view.html"
    return render(request, template, context)

### templates/ecommerce/update-view.html

In [ ]:
<h1>
    Actualización de producto {{ form.instance.title }}
</h1>

{{ form.instance.title }}

<form method="POST" action="">
    {% csrf_token %}
    {{ form.as_p }}
    <input type="submit" value="Actualizar">
</form>

### ecommerce/urls.py

In [ ]:
from django.urls import path

from ecommerce import views

urlpatterns = [
    path("", views.product_model_list_view, name="list"),
    path("<int:product_id>", views.product_model_detail_view, name="detail"),
    path("create", views.product_model_create_view, name="create"), #<----------------
    path("<int:product_id>/edit/)", views.product_model_update_view, name="update"),
    #path("redirect/", views.redirecT_to_test, name="home"),
]

## Vista para Eliminar

### ecommerce/views.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.shortcuts import render, get_object_or_404
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .forms import ProductModelForm
from .models import ProductModel

def product_model_delete_view(request, product_id):
    instance = get_object_or_404(ProductModel, id=product_id)
    if request.method == "POST":
        instance.delete()
        HttpResponseRedirect("/ecommerce/")
        messages.success("Producto eliminado")
    context = {
        "product":instance
    }
    template = "ecommerce/delete-view.html"
    return render(request, template, context)

### templates/ecommere/delete-view.html

{% include "ecommerce/messages.html" %}

<h1>
    Eliminar {{ product.title }}
</h1>

<form method="POST" action="">
    {% csrf_token %}
    Estas seguro de eliminar el producto ?
    <input type="submit" value="Eliminar">
    <a href="/ecommerce/{{ product.id }}">Cancelar</a>"
</form>

### ecommerce/urls.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.shortcuts import render, get_object_or_404
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect

from .forms import ProductModelForm
from .models import ProductModel

def product_model_delete_view(request, product_id):
    instance = get_object_or_404(ProductModel, id=product_id)
    if request.method == "POST":
        instance.delete()
        HttpResponseRedirect("/ecommerce/")
        messages.success(request, "Producto eliminado")
        return HttpResponseRedirect("/ecommerce/")
    context = {
        "product":instance
    }
    template = "ecommerce/delete-view.html"
    return render(request, template, context)

## Busqueda en la Vista de Listado

### templates/ecommerce/search.html

In [ ]:
<form method="GET" action="/ecommerce/">
    <input type="text" name="q" placeholder="Buscar">
</form>

### templates/ecomerce/list-view.html

In [ ]:
{% include "ecommerce/search.html" %}

{% include "ecommerce/messages.html" %}

<h1>
    Vista de listado
    </h1>
    
    {% for product in products %}
        <li> 
            <a href="/ecommerce/{{ product.id }}"> {{ product.title}}</a>
        </li>
    {% endfor %}

### ecommerce/views.py

In [ ]:
from django.contrib.auth.decorators import login_required
from django.contrib import messages
from django.shortcuts import render, get_object_or_404
from django.shortcuts import render
from django.http import HttpResponse, HttpResponseRedirect
from django.db.models import Q # <------------

from .forms import ProductModelForm
from .models import ProductModel

...


def product_model_list_view(request):
    query = request.GET.get("q", None)
    queryset = ProductModel.objects.all()
    if query is not None:
        queryset = queryset.filter(
            Q(title__icontains=query) |
            Q(price__icontains=query)
        )
    template = "ecommerce/list-view.html"
    context = {
        "products": queryset #<--------
    }

    if request.user.is_authenticated:
        template = "ecommerce/list-view.html"
    else:
        template = "ecommerce/list-view-public.html"

    return render(request, template, context)